# Master Recession Curve (MRC) estimation

This notebook demonstrates the Master Recession Curve (MRC) workflow implemented in `hydroevents`.

The workflow extracts recession segments from the daily streamflow series, identifies the baseflow-dominated portions of the hydrograph, and estimates the recession parameter used for baseflow separation.

## Method overview

The MRC workflow includes the following steps:

1. extraction of recession segments from the daily streamflow series;
2. identification of inflection points within recession segments;
3. extraction of baseflow-dominated segments;
4. transformation of recession segments into logarithmic space;
5. filtering of segments according to linearity criteria;
6. interpolation, local alignment of overlapping recession segments, and global alignment using a cumulative MRC fit;
7. construction of a cumulative Master Recession Curve (MRC);
8. estimation of the recession parameter using the Maillet model.

The final recession parameters are used to derive the Lyne–Hollick filter coefficient.

## Imports

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import hydroevents

from hydroevents import compute_mrc

## Load daily streamflow series

In [ ]:
file_path = Path("../examples/data/Q_daily_example.xlsx")

df_daily = pd.read_excel(file_path)

df_daily["Date"] = pd.to_datetime(df_daily["Date"])

df_daily.head()

## Define MRC parameters

The MRC workflow uses the following parameters:

- `MIN_RECESSION_LENGTH`: minimum length of decreasing discharge segments;
- `RECESSION_TOLLERANCE`:maximum allowed discharge increase (m³/s) between consecutive time steps for a segment to be considered part of the same recession period;
- `MIN_BASEFLOW_LENGTH`: minimum length of the baseflow-dominated segment after the inflection point;
- `R2_THRESHOLD`: minimum linearity threshold in logarithmic space;
- `NUM_INTERP_POINTS`: number of interpolation points used during segment alignment;
- `Q_TOLERANCE`: discharge tolerance (m3/s)  used to align overlapping recession segments.
- `SKIP_AFTER_INFLECTION`: number of time steps excluded after the inflection point before extracting the baseflow-dominated recession segment. This avoids including the transition phase immediately after the hydrograph inflection.

`SKIP_AFTER_INFLECTION` must be smaller than `MIN_BASEFLOW_LENGTH` and it must also not exceed half of `MIN_BASEFLOW_LENGTH`.

Set RECESSION_TOLERANCE > 0 to avoid splitting recession segments due to very small discharge fluctuations.

In [ ]:
MIN_RECESSION_LENGTH = 10
RECESSION_TOLERANCE = 0.0
MIN_BASEFLOW_LENGTH = 5
R2_THRESHOLD = 0.95
NUM_INTERP_POINTS = 100
Q_TOLERANCE = 0.01
SKIP_AFTER_INFLECTION = 2

## Compute MRC

The `compute_mrc()` function returns all intermediate products generated during the MRC workflow, including:

- extracted recession segments;
- baseflow-dominated segments;
- valid and discarded recession segments;
- aligned recession curves;
- cumulative MRC points;
- recession parameter estimates.

### Estimation of the recession constant
The recession constant is estimated from the slope of the fitted Maillet model in logarithmic space:

```
ln(Q) = ln(Q₀) - α t
```

where:

- `Q` is streamflow;
- `Q₀` is the initial discharge;
- `α` is the recession coefficient;
- `t` is time.

The workflow computes multiple recession constant estimates using different approaches:

- `mrc`: derived from the final cumulative Master Recession Curve fit;
- `mean`: computed as the mean recession coefficient of valid individual segments;
- `median`: computed as the median recession coefficient of valid individual segments.

The `mrc` estimate is generally considered the most representative because it is derived from the cumulative aligned recession behaviour of all valid segments.

The estimated recession coefficient is then used to derive the Lyne–Hollick filter parameter:

For daily time steps:

```
k_day = exp(-α)
```

For hourly time steps:

```
k_hour = exp(-α / 24)
```

where `α` is estimated from the daily Master Recession Curve.

Both daily (`k_day`) and hourly (`k_hour`) filter parameters are computed and returned by the workflow for subsequent baseflow separation analyses.

In [ ]:
mrc_results = compute_mrc(
    df_daily,
    min_recession_length=MIN_RECESSION_LENGTH,
    recession_tolerance=RECESSION_TOLERANCE,
    min_baseflow_length=MIN_BASEFLOW_LENGTH,
    skip_after_inflection=SKIP_AFTER_INFLECTION,
    r2_threshold=R2_THRESHOLD,
    num_interp_points=NUM_INTERP_POINTS,
    q_tolerance=Q_TOLERANCE,
)

mrc_results.keys()

## Recession segment statistics

In [ ]:
print("Extracted recession segments:", len(mrc_results["recession_segments"]))
print("Baseflow segments:", len(mrc_results["baseflow_results"]))
print("Valid segments:", len(mrc_results["valid_segments"]))
print("Discarded segments:", len(mrc_results["discarded_segments"]))

## Extracted recession segments

Recession segments correspond to periods of continuously decreasing streamflow identified from the daily hydrograph.

In [ ]:
recession_segments = mrc_results["recession_segments"]

fig = go.Figure()

for i, seg in enumerate(recession_segments):
    fig.add_trace(
        go.Scatter(
            x=seg["Date"],
            y=seg["Q"],
            mode="lines",
            name=f"Segment {i + 1}",
            showlegend=False,
        )
    )

fig.update_layout(
    title="Extracted recession segments",
    xaxis_title="Date",
    yaxis_title="Q",
    template="plotly_white",
)

fig.show()

## Baseflow-dominated recession segments

For each recession segment, an inflection point is identified using the second derivative of the hydrograph.

Only the recession portion after the inflection point is retained as the baseflow-dominated segment used for MRC construction.

In [ ]:
baseflow_results = mrc_results["baseflow_results"]

fig = go.Figure()

for i, res in enumerate(baseflow_results):

    seg = res["baseflow_segment"]
    infl = res["inflection_point"]

    # baseflow-dominated segment
    fig.add_trace(
        go.Scatter(
            x=seg["Date"],
            y=seg["Q"],
            mode="lines",
            name=f"Baseflow segment {i + 1}",
            showlegend=False,
        )
    )

    # inflection point
    fig.add_trace(
        go.Scatter(
            x=infl["Date"],
            y=infl["Q"],
            mode="markers",
            marker=dict(size=7),
            name="Inflection point" if i == 0 else None,
            showlegend=(i == 0),
        )
    )

fig.update_layout(
    title="Baseflow-dominated recession segments",
    xaxis_title="Date",
    yaxis_title="Q",
    template="plotly_white",
)

fig.show()

## Master Recession Curve fit

The cumulative MRC points are fitted in logarithmic space using the Maillet recession model.  
The slope of the fitted line provides the recession coefficient `α`.

In [ ]:
df_mrc = mrc_results["df_mrc"].copy()
fit_results = mrc_results["fit_results"]

alpha = fit_results["alpha"]
ln_q0 = fit_results["ln_Q0"]
r2 = fit_results["R2"]

df_mrc["ln_Q_fit"] = ln_q0 - alpha * df_mrc["t_global"]

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df_mrc["t_global"],
        y=df_mrc["ln_Q"],
        mode="markers",
        name="MRC points",
    )
)

fig.add_trace(
    go.Scatter(
        x=df_mrc["t_global"],
        y=df_mrc["ln_Q_fit"],
        mode="lines",
        name="Maillet fit",
    )
)

fig.update_layout(
    title=f"Final MRC fit | α = {alpha:.4f} day⁻¹ | R² = {r2:.3f}",
    xaxis_title="Aligned time [days]",
    yaxis_title="ln(Q)",
    template="plotly_white",
)

fig.show()

## Recession parameter estimates

The table below summarizes the estimated filter parameter (`k`) obtained using the different MRC approaches.

In [ ]:
mrc_results["df_k_estimates"]

## Save output

The main MRC outputs can be exported for subsequent analyses and reproducibility purposes.

In [ ]:
output_dir = Path(r"path\to\output")

output_dir.mkdir(parents=True, exist_ok=True)

# cumulative MRC points
mrc_results["df_mrc"].to_excel(
    output_dir / "mrc_points.xlsx",
    index=False,
)

# recession parameter estimates
mrc_results["df_k_estimates"].to_excel(
    output_dir / "mrc_k_estimates.xlsx",
    index=False,
)

# globally shifted recession segments
mrc_results["df_shifted_segments"].to_excel(
    output_dir / "mrc_shifted_segments.xlsx",
    index=False,
)

# alignment diagnostics
mrc_results["df_alignment_log"].to_excel(
    output_dir / "mrc_alignment_log.xlsx",
    index=False,
)

print(f"Results saved to: {output_dir.resolve()}")